In [ ]:
import json
import math
import statistics
from collections import Counter, defaultdict
from dataclasses import dataclass
from datetime import datetime
from typing import Any, Dict, List, Optional, Tuple

In [ ]:

def load_json(path: str) -> Dict[str, Any]:
    with open(path, 'r', encoding='utf-8') as f:
        return json.load(f)

In [ ]:
def save_json(path: str, data: Dict[str, Any]) -> None:
    with open(path, 'w', encoding='utf-8') as f:
        json.dump(data, f, ensure_ascii=False, indent=2)

In [ ]:
def to_float(value: Any) -> Optional[float]:
    if value is None or value == "":
        return None
    if isinstance(value, (int, float)):  # Se já for número (int ou float), apenas converte para float (padroniza o tipo).
        return float(value)
    if isinstance(value, str):           # Se for string, entra no processamento mais interessante
        clean = value.strip().replace("R$", "").replace(".", "").replace(",", ".")
        try:
            return float(clean)
        except ValueError:
            return None
    return None

In [ ]:

def to_bool(value: Any) -> Optional[bool]:
    if value is None or value == "":
        return None
    if isinstance(value, bool):  # Se já for booleano (True ou False), retorna direto.
        return value
    if isinstance(value, (int, float)):  
        return bool(value)
    if isinstance(value, str):         # Se for string: remove espaços (strip) e converte para minúsculas (lower)
        text = value.strip().lower()
        if text in {"true", "1", "sim", "s", "yes", "y"}:
            return True
        if text in {"false", "0", "nao", "não", "n", "no"}:
            return False
    return None

In [ ]:

def parse_date(value: Any) -> Optional[str]:
    if value is None or value == "":
        return None
    if isinstance(value, str):  # Só tenta converter se o valor for uma string.
        for fmt in ("%Y-%m-%d", "%d/%m/%Y", "%Y/%m/%d", "%d-%m-%Y"):
            try:
                return datetime.strptime(value, fmt).strftime("%Y-%m-%d")
            except ValueError:
                continue
        return value
    return None

In [ ]:

def safe_mean(values: List[float]) -> Optional[float]:
    return statistics.mean(values) if values else None

In [ ]:
def safe_median(values: List[float]) -> Optional[float]:
    return statistics.median(values) if values else None

In [ ]:
def safe_mode(values: List[float]) -> Optional[float]:
    if not values:   # Se a lista estiver vazia, retorna None.
        return None
    counts = Counter(values)          # Usa Counter (da biblioteca collections) para contar quantas vezes cada valor aparece.
    top_count = max(counts.values())  # Encontra a maior frequência (quantas vezes o valor mais comum aparece).
    top_values = [k for k, v in counts.items() if v == top_count]
    return top_values[0] if top_values else None   # se der empate, retorna apenas o primeiro valor da lista

In [ ]:
def safe_variance(values: List[float]) -> Optional[float]:
    return statistics.variance(values) if len(values) > 1 else 0.0 if values else None

In [ ]:

def safe_std(values: List[float]) -> Optional[float]:
    return statistics.stdev(values) if len(values) > 1 else 0.0 if values else None

In [ ]:
def safe_min(values: List[float]) -> Optional[float]:
    return min(values) if values else None

In [ ]:
def safe_max(values: List[float]) -> Optional[float]:
    return max(values) if values else None

In [ ]:
def amplitude(values: List[float]) -> Optional[float]:
    return (max(values) - min(values)) if values else None

In [ ]:

def percentile(values: List[float], p: float) -> Optional[float]:
    if not values:
        return None
    ordered = sorted(values)
    if len(ordered) == 1:
        return ordered[0]
    idx = (len(ordered) - 1) * p
    lo = math.floor(idx)
    hi = math.ceil(idx)
    if lo == hi:
        return ordered[int(idx)]
    fraction = idx - lo
    return ordered[lo] + (ordered[hi] - ordered[lo]) * fraction

In [ ]:

def quartiles(values: List[float]) -> Dict[str, Optional[float]]:
    return {
        "q1": percentile(values, 0.25),
        "q2_mediana": percentile(values, 0.50),
        "q3": percentile(values, 0.75),
    }

In [ ]:
def deciles(values: List[float]) -> Dict[str, Optional[float]]:
    return {f"d{i}": percentile(values, i / 10) for i in range(1, 10)}

In [ ]:

def percentiles_selected(values: List[float]) -> Dict[str, Optional[float]]:
    return {
        "p10": percentile(values, 0.10),
        "p25": percentile(values, 0.25),
        "p50": percentile(values, 0.50),
        "p75": percentile(values, 0.75),
        "p90": percentile(values, 0.90),
    }

In [ ]:
def frequency_table(values: List[Any], top_n: int = 10) -> List[Dict[str, Any]]:
    counts = Counter(v for v in values if v is not None and v != "")
    total = sum(counts.values())
    table = []
    for item, count in counts.most_common(top_n):
        table.append({
            "valor": item,
            "frequencia": count,
            "percentual": round((count / total) * 100, 2) if total else 0.0,
        })
    return table

In [ ]:
def contingency_table(rows: List[Any], cols: List[Any], top_n: int = 10) -> Dict[str, Dict[str, int]]:
    result: Dict[str, Dict[str, int]] = defaultdict(lambda: defaultdict(int))
    for r, c in zip(rows, cols):
        if r is None or c is None or r == "" or c == "":
            continue
        result[str(r)][str(c)] += 1  # Conta quantas vezes o par (r, c) aparece
    limited = {}
    for i, (rk, rv) in enumerate(result.items()):
        if i >= top_n:
            break
        limited[rk] = dict(list(rv.items())[:top_n])
    return limited

In [ ]:
def pearson_correlation(x: List[float], y: List[float]) -> Optional[float]:
    if len(x) != len(y) or len(x) < 2:
        return None
    mx = safe_mean(x)  # Média de x
    my = safe_mean(y)  # Média de y 
    sx = safe_std(x)   # Desvio padrão de x
    sy = safe_std(y)   # Desvio padrão de y
    if mx is None or my is None or sx in (None, 0) or sy in (None, 0):
        return None
    numerator = sum((a - mx) * (b - my) for a, b in zip(x, y))
    denominator = (len(x) - 1) * sx * sy
    if denominator == 0:  # Evita divisão por zero
        return None
    return numerator / denominator  # Resultado final da correlação de Pearson

In [ ]:
def min_max_normalize(values: List[float]) -> List[Optional[float]]:
    if not values:
        return []
    lo, hi = min(values), max(values)  # Encontra o menor (lo) e o maior (hi) valor da lista
    if hi == lo:  # Se todos os valores são iguais, não há variação
        return [0.0 for _ in values] # Retorna lista de zeros (todos iguais após normalização)
    return [(v - lo) / (hi - lo) for v in values]

In [ ]:
def z_score_standardize(values: List[float]) -> List[Optional[float]]:
    if not values:  # Se a lista estiver vazia, não há o que calcular
        return []
    mu = safe_mean(values)     # Calcula a média (μ)
    sigma = safe_std(values)   # Calcula o desvio padrão (σ)
    if mu is None:             # Se não foi possível calcular a média, retorna vazio
        return []
    if sigma in (None, 0):     # Se não há variação (desvio padrão zero), todos os valores são iguais
        return [0.0 for _ in values]  # Nesse caso, todos recebem 0
    return [(v - mu) / sigma for v in values]

In [ ]:
def iqr_outliers(values: List[float]) -> Dict[str, Any]:
    if len(values) < 4:  # Com poucos dados, o cálculo de quartis não é confiável
        return {
            "q1": percentile(values, 0.25),  # Primeiro quartil (25%)
            "q3": percentile(values, 0.75),  # Terceiro quartil (75%)
            "iqr": None,                     # Não dá para calcular bem o IQR 
            "limite_inferior": None,         # Sem limite inferior
            "limite_superior": None,         # Sem limite superior
            "indices_outliers": [],          # Nenhum outlier identificado
            "valores_outliers": [],          # Nenhum valor fora do padrão
        }
    q1 = percentile(values, 0.25)     # Calcula o primeiro quartil
    q3 = percentile(values, 0.75)     # Calcula o terceiro quartil
    assert q1 is not None and q3 is not None   # Garante que os valores não são None (checagem de segurança)
    iqr = q3 - q1                     # IQR = intervalo entre Q3 e Q1 (parte central dos dados)
    low = q1 - 1.5 * iqr              # Limite inferior para detectar outliers
    high = q3 + 1.5 * iqr             # Limite superior para detectar outliers
    indices = [i for i, v in enumerate(values) if v < low or v > high]  # Encontra índices dos valores fora dos limites
    out_vals = [values[i] for i in indices]  # Pega os valores reais dos outliers
    return {
        "q1": q1,    # resumo estatístico
        "q3": q3,
        "iqr": iqr,
        "limite_inferior": low,   # limites para considerar outliers
        "limite_superior": high,
        "indices_outliers": indices,  # posições dos outliers na lista
        "valores_outliers": out_vals,  # valores que são outliers
    }

In [ ]:

def null_report(records: List[Dict[str, Any]]) -> Dict[str, int]:
    counts: Dict[str, int] = Counter()  # counts deve se comportar como dicionário com chaves do tipo str e valores do tipo int
    for rec in records:   # Percorre cada registro (cada "linha" de dados)
        for k, v in rec.items():  # Percorre cada campo (chave e valor) dentro do registro
            if v is None or v == "":  # Verifica se o valor está vazio (None ou string vazia)
                counts[k] += 1    # Incrementa 1 na contagem daquela chave
    return dict(counts)   # Converte o Counter para dict normal e retorna

In [ ]:
def infer_field_type(values: List[Any]) -> str:
    not_null = [v for v in values if v is not None and v != ""] # Remove valores vazios (None ou "")
    if not not_null:          # Se não sobrou nenhum valor válido
        return "indefinido"
    numeric = sum(1 for v in not_null if isinstance(v, (int, float)) or to_float(v) is not None)
    if numeric == len(not_null):  # Se TODOS os valores são numéricos
        unique = len(set(float(to_float(v)) for v in not_null if to_float(v) is not None)) # Conta quantos valores numéricos diferentes existem
        if unique <= 12:   # Poucos valores distintos → provavelmente discreto
            return "quantitativa_discreta"
        return "quantitativa_continua"  # Muitos valores distintos → contínuo (ex: salário, peso)
    unique = len(set(str(v) for v in not_null))  # Se não for totalmente numérico, trata como texto e conta categorias únicas
    if unique <= 12:  # Poucas categorias → variável categórica simples
        return "qualitativa_nominal"
    return "qualitativa_ordinal_ou_textual"  # Muitas categorias → texto livre ou variável ordenada complexa

In [ ]:
def round_or_none(value: Optional[float], ndigits: int = 4) -> Optional[float]:
    return round(value, ndigits) if isinstance(value, (int, float)) else value

In [ ]:

def round_list(values: List[Optional[float]], ndigits: int = 4) -> List[Optional[float]]:
    return [round_or_none(v, ndigits) for v in values]

In [ ]:

def normal_loglik(values: List[float]) -> Optional[float]:
    if not values:   # Se não há dados, não dá para calcular
        return None
    mu = safe_mean(values)     # Média dos valores (μ)
    sigma = safe_std(values)   # Desvio padrão (σ)
    if mu is None or sigma in (None, 0):  # Sem média ou sem variação → não dá para modelar normal
        return None  # Soma da log-verossimilhança de cada ponto
    return sum(-0.5 * math.log(2 * math.pi * sigma ** 2) - ((x - mu) ** 2) / (2 * sigma ** 2) for x in values)

In [ ]:
def exponential_loglik(values: List[float]) -> Optional[float]:
    if not values or any(v <= 0 for v in values): # Se não há dados ou existe valor <= 0, não faz sentido na distribuição exponencial
        return None
    mu = safe_mean(values)   # Calcula a média dos valores
    if mu in (None, 0):      # Média inválida ou zero impede cálculo do parâmetro
        return None          # λ (lambda) é o parâmetro da distribuição exponencial
    lam = 1 / mu
    return sum(math.log(lam) - lam * x for x in values)

In [ ]:

def uniform_loglik(values: List[float]) -> Optional[float]:
    if not values:   # Se não há dados, não dá para calcular
        return None
    lo = min(values) # Menor valor da amostra (limite inferior do intervalo)
    hi = max(values) # Maior valor da amostra (limite superior do intervalo)
    if hi == lo:     # Se todos os valores são iguais, não existe intervalo
        return None
    return len(values) * math.log(1 / (hi - lo))

In [ ]:
def poisson_loglik(counts: List[int]) -> Optional[float]:
    if not counts or any(c < 0 for c in counts):  # Poisson só aceita contagens (0, 1, 2, 3...)
        return None
    lam = safe_mean([float(c) for c in counts])
    if lam is None or lam <= 0: # λ precisa ser positivo
        return None
    total = 0.0                 # acumulador da log-verossimilhança
    for k in counts:            # para cada contagem observada
        total += k * math.log(lam) - lam - math.lgamma(k + 1)
    return total # retorna soma total da log-verossimilhança

In [ ]:
def bernoulli_loglik(values: List[int]) -> Optional[float]: # Calcula log-verossimilhança assumindo distribuição de Bernoulli
    if not values or any(v not in (0, 1) for v in values):  # Bernoulli só aceita valores 0 ou 1
        return None
    p = safe_mean([float(v) for v in values])   
    if p is None or p in (0, 1):    # p não pode ser 0 ou 1 porque geraria log(0)
        return None
    return sum(v * math.log(p) + (1 - v) * math.log(1 - p) for v in values)

In [ ]:
def binomial_loglik(values: List[int], n: Optional[int] = None) -> Optional[float]:
    if not values or any(v < 0 for v in values):  # Não aceita lista vazia nem valores negativos (não faz sentido em contagem de sucessos)
        return None
    if n is None:
        n = max(values) if values else 0
    if n <= 0:
        return None
    mean_val = safe_mean([float(v) for v in values])  # Média dos sucessos observados
    if mean_val is None:
        return None
    p = mean_val / n      # Estima a probabilidade de sucesso p = média / número de tentativas
    if p <= 0 or p >= 1:  # p precisa estar entre 0 e 1 (exclusivo)
        return None
    total = 0.0           # acumulador da log-verossimilhança
    for k in values:      # k = número de sucessos observados
        if k > n:         # não pode ter mais sucessos que tentativas
            return None
        total += math.lgamma(n + 1) - math.lgamma(k + 1) - math.lgamma(n - k + 1)
        total += k * math.log(p) + (n - k) * math.log(1 - p)
    return total

In [ ]:

@dataclass
class PreparedData:
    negocio: Dict[str, Any]
    transacoes: List[Dict[str, Any]]
    dias: List[Dict[str, Any]]
    recepcao: Dict[str, Any]

In [ ]:
class InsightCalculadoEngine:
    def __init__(self, data: Dict[str, Any]):
        self.raw = data                            # guarda os dados originais
        self.prepared = self._prepare_data(data)   # chama o método da própria classe e salva o resultado em prepared

In [ ]:
    def _prepare_data(self, data: Dict[str, Any]) -> PreparedData:
        negocio = data.get("negocio", {})
        dados = data.get("dados", {})
        recepcao = data.get("recepcao", {})

In [ ]:
        transacoes = []   # criando lista vazia
        for item in dados.get("transacoes",[]):     # [] retorna uma lista vazia
            record = dict(item)                     # cria um dicionário (dict) a partir de item.
            record["data"] = parse_date(record.get("data"))  # trata o formato de data
            if "valor" in record:
                record["valor"] = to_float(record.get("valor"))     # trata float
            if "pago_no_prazo" in record:
                record["pago_no_prazo"] = to_bool(record.get("pago_no_prazo"))   # trata boolean
            if "desconto" in record:
                record["desconto"] = to_float(record.get("desconto"))    # trata float
            if "marketing" in record:
                record["marketing"] = to_float(record.get("marketing"))  # trata float
            transacoes.append(record)     # adicionando na lista

In [ ]:
        dias = []    # criando lista vazia
        for item in dados.get("dias", []):
            record = dict(item)
            record["data"] = parse_date(record.get("data"))
            for field in ["receita", "despesa", "vendas_qtd", "clientes", "marketing", "desconto_medio"]:
                if field in record:  # verifica se a chave field existe dentro do dicionário record
                    record[field] = to_float(record.get(field))   # trata float
            dias.append(record)

In [ ]:
        if not dias and transacoes:
            dias = self._derive_daily_records(transacoes)

In [ ]:
        return PreparedData(negocio=negocio, transacoes=transacoes, dias=dias, recepcao=recepcao)

In [ ]:
    def _derive_daily_records(self, transacoes: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
        by_day: Dict[str, Dict[str, Any]] = defaultdict(lambda: {
            "receita": 0.0,
            "despesa": 0.0,
            "vendas_qtd": 0,
            "clientes": set(),
            "marketing": 0.0,
            "desconto_medio_soma": 0.0,
            "desconto_medio_qtd": 0,
            "transacoes_qtd": 0,
        })

In [ ]:
        for t in transacoes:
            day = t.get("data") or "sem_data"
            info = by_day[day]
            info["transacoes_qtd"] += 1
            valor = t.get("valor") or 0.0
            tipo = str(t.get("tipo", "")).lower()
            if tipo == "receita":
                info["receita"] += valor
                info["vendas_qtd"] += 1
            elif tipo == "despesa":
                info["despesa"] += valor
            info["marketing"] += (t.get("marketing") or 0.0)
            desconto = t.get("desconto")
            if desconto is not None:
                info["desconto_medio_soma"] += desconto
                info["desconto_medio_qtd"] += 1
            cliente = t.get("cliente")
            if cliente:
                info["clientes"].add(str(cliente))

In [ ]:
        daily_records = []
        for day, info in sorted(by_day.items(), key=lambda x: x[0]):
            qtd_desc = info["desconto_medio_qtd"]
            daily_records.append({
                "data": day,
                "receita": round(info["receita"], 2),
                "despesa": round(info["despesa"], 2),
                "vendas_qtd": int(info["vendas_qtd"]),
                "clientes": int(len(info["clientes"])),
                "marketing": round(info["marketing"], 2),
                "desconto_medio": round((info["desconto_medio_soma"] / qtd_desc), 4) if qtd_desc else 0.0,
                "transacoes_qtd": int(info["transacoes_qtd"]),
            })
        return daily_records

In [ ]:
    def _series_from_days(self) -> Dict[str, List[float]]:
        dias = self.prepared.dias  # Obtém a lista de dias já preparados (provavelmente um array de dicionários com métricas diárias)
        receita = [d.get("receita") for d in dias if d.get("receita") is not None]
        despesa = [d.get("despesa") for d in dias if d.get("despesa") is not None]
        lucro = []        # Inicializa a lista de lucro (será calculada manualmente)
        for d in dias:    # Itera sobre cada dia
            if d.get("receita") is not None and d.get("despesa") is not None:
                lucro.append(float(d["receita"]) - float(d["despesa"])) # Só calcula lucro se ambos os valores existirem
        vendas_qtd = [int(d.get("vendas_qtd") or 0) for d in dias]
        clientes = [float(d.get("clientes") or 0) for d in dias]
        marketing = [float(d.get("marketing") or 0) for d in dias]
        desconto_medio = [float(d.get("desconto_medio") or 0) for d in dias]
        transacoes_qtd = [int(d.get("transacoes_qtd") or d.get("vendas_qtd") or 0) for d in dias]
        return {                           # Retorna um dicionário contendo todas as séries organizadas por tipo
            "receita": receita,
            "despesa": despesa,
            "lucro": lucro,
            "vendas_qtd": vendas_qtd,
            "clientes": clientes,
            "marketing": marketing,
            "desconto_medio": desconto_medio,
            "transacoes_qtd": transacoes_qtd,
        }

In [ ]:
    def _ticket_values(self) -> List[float]:
        tickets = []  # Inicializa uma lista vazia que armazenará os valores dos tickets (receitas)
        for t in self.prepared.transacoes:   # Percorre cada transação presente na estrutura preparada
            if str(t.get("tipo", "")).lower() == "receita" and t.get("valor") is not None:
                tickets.append(float(t["valor"]))  # Converte o valor da transação para float e adiciona à lista
        return tickets  # Retorna a lista contendo todos os valores de tickets encontrados

In [ ]:

    def aula_1(self) -> Dict[str, Any]:   # Define o método "aula_1", que retorna um dicionário com análises iniciais dos dados
        transacoes = self.prepared.transacoes   # Extrai a lista de transações já preparadas
        dias = self.prepared.dias               # Extrai a lista de registros diários já preparados
        if transacoes:                          # Verifica se existem transações
            keys = sorted({k for rec in transacoes for k in rec.keys()})      # sorted transforma o set em uma lista ordenada
            classificacao = {}    # Inicializa dicionário que armazenará o tipo de cada campo
            for key in keys:      # Itera sobre cada campo existente nas transações
                valores = [rec.get(key) for rec in transacoes]   # Coleta todos os valores daquele campo em todas as transações
                classificacao[key] = infer_field_type(valores)
        else:   # Caso não existam transações
            classificacao = {}    # Define classificação vazia

In [ ]:
        sample_size = min(5, len(transacoes))   # Define tamanho da amostra (no máximo 5 registros)
        amostra = transacoes[:sample_size]      # Pega os primeiros registros como amostra

In [ ]:
        return {  # Retorna um relatório estruturado com análises da "aula 1"
            "tema": "Entender os dados do negócio",  # Define o objetivo da análise
            "problema_financeiro": "O empreendedor possui dados, mas não sabe o que está registrando nem como organizar isso para análise.",
            "calculos": {  # Seção com métricas e cálculos
                "populacao_transacoes": len(transacoes),    # Quantidade total de transações
                "populacao_registros_diarios": len(dias),   # Quantidade total de registros diários
                "amostra_exibida": sample_size,             # Quantidade de registros mostrados na amostra
                "classificacao_campos_transacoes": classificacao,        # Tipos de cada campo das transações
                "campos_faltantes_transacoes": null_report(transacoes),  # Conta valores nulos/vazios nas transações
                "campos_faltantes_dias": null_report(dias),  # Conta valores nulos/vazios nos dados diários
                "amostra_transacoes": amostra,               # Exibe amostra dos dados
            },
            "insights": [ # Lista de interpretações automáticas
                "Nesta etapa o sistema identifica quais campos são qualitativos e quais são quantitativos.",
                "Também aponta lacunas iniciais para preparar a análise exploratória das próximas aulas.",
            ],
        }

In [ ]:
    def aula_2(self) -> Dict[str, Any]:    # Define o método aula_2, que retorna um relatório estatístico do negócio

In [ ]:

        series = self._series_from_days()
        tickets = self._ticket_values()     # Extrai valores de vendas (receitas) para análise de ticket

In [ ]:
        def resumo(values: List[float]) -> Dict[str, Optional[float]]:  # Função interna que calcula estatísticas descritivas de uma lista
            return {
                "media": round_or_none(safe_mean(values), 4),       # Calcula a média dos valores e arredonda
                "mediana": round_or_none(safe_median(values), 4),   # Calcula a mediana (valor central) 
                "moda": round_or_none(safe_mode(values), 4),        # Calcula a moda (valor mais frequente)
                "amplitude": round_or_none(amplitude(values), 4),   # Diferença entre máximo e mínimo
                "variancia": round_or_none(safe_variance(values), 4),  # Mede dispersão dos dados
                "desvio_padrao": round_or_none(safe_std(values), 4),   # Mede o quanto os valores variam em torno da média
                "minimo": round_or_none(safe_min(values), 4),       # Menor valor da lista
                "maximo": round_or_none(safe_max(values), 4),       # Maior valor da lista
            }

In [ ]:
        return {# Retorna um relatório estruturado da aula 2
            "tema": "Descobrir o comportamento financeiro central do negócio",  # Define o objetivo da análise
            "problema_financeiro": "O empreendedor não sabe qual é o comportamento financeiro típico do negócio.",
            "calculos": { # Bloco com cálculos estatísticos principais
                "receita_diaria": resumo(series["receita"]),   # Estatísticas da receita por dia
                "despesa_diaria": resumo(series["despesa"]),   # Estatísticas da despesa por dia
                "lucro_diario": resumo(series["lucro"]),       # Estatísticas do lucro por dia
                "ticket_receita": resumo(tickets),             # Estatísticas dos valores de venda (ticket médio)
            },
            "insights": [  # Interpretações automáticas dos dados
                "A média mostra o comportamento central, enquanto mediana e moda ajudam a validar se há distorções.",
                "A variância e o desvio padrão ajudam a medir estabilidade financeira.",
            ],
        }

In [ ]:
    def aula_3(self) -> Dict[str, Any]:
        transacoes = self.prepared.transacoes
        series = self._series_from_days()
        tickets = self._ticket_values()
        categorias = [t.get("categoria") for t in transacoes]
        formas_pagamento = [t.get("forma_pagamento") for t in transacoes]

In [ ]:
        return {
            "tema": "Organizar os dados para entender distribuição e faixas de desempenho",
            "problema_financeiro": "O empreendedor vê números soltos, mas não enxerga faixas de valor, frequência ou concentração.",
            "calculos": {
                "tabela_frequencia_categoria": frequency_table(categorias),
                "tabela_frequencia_forma_pagamento": frequency_table(formas_pagamento),
                "tabela_contingencia_categoria_pagamento": contingency_table(categorias, formas_pagamento),
                "quartis_receita_diaria": {k: round_or_none(v, 4) for k, v in quartiles(series["receita"]).items()},
                "quartis_lucro_diario": {k: round_or_none(v, 4) for k, v in quartiles(series["lucro"]).items()},
                "decis_ticket": {k: round_or_none(v, 4) for k, v in deciles(tickets).items()},
                "percentis_ticket": {k: round_or_none(v, 4) for k, v in percentiles_selected(tickets).items()},
            },
            "insights": [
                "As tabelas ajudam a descobrir onde há maior concentração de receitas, despesas e formas de pagamento.",
                "Quartis, decis e percentis permitem segmentar clientes, tickets e períodos do negócio.",
            ],
        }

In [ ]:
    def aula_4(self) -> Dict[str, Any]:
        series = self._series_from_days()
        corr_pairs = {
            "marketing_x_receita": pearson_correlation(series["marketing"], series["receita"]),
            "marketing_x_lucro": pearson_correlation(series["marketing"], series["lucro"]),
            "desconto_x_receita": pearson_correlation(series["desconto_medio"], series["receita"]),
            "clientes_x_receita": pearson_correlation(series["clientes"], series["receita"]),
            "vendas_qtd_x_receita": pearson_correlation([float(v) for v in series["vendas_qtd"]], series["receita"]),
            "despesa_x_lucro": pearson_correlation(series["despesa"], series["lucro"]),
        }
        rounded = {k: round_or_none(v, 4) for k, v in corr_pairs.items()}
        valid = {k: v for k, v in rounded.items() if v is not None}
        strongest = None
        if valid:
            strongest = max(valid.items(), key=lambda item: abs(item[1]))

In [ ]:
        return {
            "tema": "Entender o que influencia o resultado financeiro",
            "problema_financeiro": "O empreendedor quer saber o que parece impactar faturamento, lucro ou estabilidade.",
            "calculos": {
                "correlacoes": rounded,
                "relacao_mais_forte": {
                    "par": strongest[0],
                    "correlacao": strongest[1],
                } if strongest else None,
                "nota_causalidade": "Correlação não prova causalidade; os resultados indicam associação estatística e devem ser validados no contexto do negócio.",
            },
            "insights": [
                "Esta etapa ajuda a entender o que se move junto com receita ou lucro.",
                "Ela não substitui análise causal, mas orienta testes e decisões do projeto.",
            ],
        }

In [ ]:
if __name__ == "__main__":  # se o arquivo estiver sendo executado diretamente

In [ ]:
    dados1=load_json("teste.json")    # carregando o arquivo json em um dicionário
    dados2=load_json("teste2.json")
    dados3=load_json("teste3.json") 

In [ ]:
    print(dados1["nome"])    # exibindo na tela

In [ ]:
    print(dados2["empresa"]["nome"])

In [ ]:
    print(dados2["clientes"])

In [ ]:
    print(dados2["clientes"][0])  # acessando um item da lista

In [ ]:
    print(dados2["clientes"][1])

In [ ]:
    print(dados2["clientes"][1]["nome"])

In [ ]:
    print(dados3["clientes"][0]["compras"][0]["produto"])

In [ ]:

    print(to_float("teste"))
    print(to_float("R$ 100,00"))
    print(to_float(34.5))

In [ ]:

    print(to_bool("teste"))
    print(to_bool(True))
    print(to_bool(False))
    print(to_bool("true"))
    print(to_bool("false"))
    print(to_bool("yes"))
    print(to_bool("no"))

In [ ]:

    print("Teste 1 - None")
    print(parse_date(None))  

In [ ]:
    print("\nTeste 2 - string vazia")
    print(parse_date(""))  

In [ ]:
    print("\nTeste 3 - formato ISO (2025-04-25)")
    print(parse_date("2025-04-25"))  

In [ ]:
    print("\nTeste 4 - formato brasileiro (25/04/2025)")
    print(parse_date("25/04/2025"))  

In [ ]:
    print("\nTeste 5 - formato com barras (2025/04/25)")
    print(parse_date("2025/04/25"))  

In [ ]:
    print("\nTeste 6 - formato com hífen (25-04-2025)")
    print(parse_date("25-04-2025"))  

In [ ]:
    print("\nTeste 7 - string inválida")
    print(parse_date("data errada"))  

In [ ]:
    print("\nTeste 8 - número (tipo inválido)")
    print(parse_date(123))  

In [ ]:

    print("Teste 1 - lista vazia")
    print(infer_field_type([]))

In [ ]:
    print("\nTeste 2 - todos vazios")
    print(infer_field_type([None, "", None]))

In [ ]:
    print("\nTeste 3 - números inteiros (poucos valores distintos)")
    print(infer_field_type([1, 2, 3, 4, 5]))

In [ ]:
    print("\nTeste 4 - números repetidos (discreto também)")
    print(infer_field_type([10, 10, 10, 10]))

In [ ]:
    print("\nTeste 5 - números com muitos valores diferentes")
    print(infer_field_type([1.1, 2.5, 3.7, 4.2, 5.9, 6.3, 7.8, 8.1, 9.4, 10.6, 11.2, 12.9, 13.3]))

In [ ]:
    print("\nTeste 6 - valores como string numérica")
    print(infer_field_type(["10", "20", "30"]))

In [ ]:
    print("\nTeste 7 - categorias pequenas")
    print(infer_field_type(["sim", "não", "sim", "não"]))

In [ ]:
    print("\nTeste 8 - muitas categorias diferentes")
    print(infer_field_type(["a", "b", "c", "d", "e", "f", "g", "h", "i", "j", "k", "l", "m"]))

In [ ]:
    print("\nTeste 9 - mistura inválida (texto + números)")
    print(infer_field_type([10, "20", "abc", None]))

In [ ]:

    print("Teste 1 - lista vazia")
    print(null_report([]))

In [ ]:
    print("\nTeste 2 - sem valores nulos")
    dados = [
        {"nome": "Ana", "idade": 28},
        {"nome": "Carlos", "idade": 35}
    ]
    print(null_report(dados))

In [ ]:
    print("\nTeste 3 - alguns valores nulos")
    dados = [
        {"nome": "Ana", "idade": None},
        {"nome": "", "idade": 35},
        {"nome": "Carlos", "idade": None}
    ]
    print(null_report(dados))

In [ ]:
    print("\nTeste 4 - todos os campos nulos")
    dados = [
        {"nome": None, "idade": None},
        {"nome": "", "idade": None}
    ]
    print(null_report(dados))

In [ ]:
    print("\nTeste 5 - campos diferentes por registro")
    dados = [
        {"nome": "Ana", "idade": None},
        {"nome": None},
        {"idade": ""}
    ]
    print(null_report(dados))

In [ ]:
    print("\nTeste 6 - mistura de valores válidos e inválidos")
    dados = [
        {"produto": "Mouse", "preco": 50},
        {"produto": "", "preco": None},
        {"produto": "Teclado", "preco": 100},
        {"produto": None, "preco": ""}
    ]
    print(null_report(dados))

In [ ]:

    engine = InsightCalculadoEngine({})

In [ ]:
    data = {
        "negocio": {"nome": "Loja Teste"},
        "dados": {
            "transacoes": [
                {
                    "data": "01/01/2024",
                    "valor": "R$ 100,50",
                    "pago_no_prazo": "sim",
                    "desconto": "10",
                    "marketing": "5"
                }
            ],
            "dias": [
                {
                    "data": "01/01/2024",
                    "receita": "100",
                    "despesa": "50",
                    "vendas_qtd": "2",
                    "clientes": "3",
                    "marketing": "5",
                    "desconto_medio": "10"
                }
            ]
        },
        "recepcao": {"origem": "teste"}
    }

In [ ]:
    resultado = engine._prepare_data(data)  # Crie um motor (engine) usando dados vazios. Usamos {} porque

In [ ]:
    print("\nTeste 1 - Dados completos")
    print(resultado)

In [ ]:

    dados_empresa = load_json("exemplo_entrada_insight_calculado.json")
    engine = InsightCalculadoEngine(dados_empresa)
    resultado = engine.aula_4()             
    print("\nTeste 4 - aula4")
    print(resultado)
    save_json("teste_saida_aula4.json", resultado)